# 工作记忆与上下文窗口管理

> 主动地对上下文元素排定优先级、固定和淘汰，让最相关的信息占据有限的上下文窗口。

想象你是一家忙碌餐厅的服务员。你脑子里一次最多能记住四份订单。第五桌招手叫你时，你必须写下点什么，否则就得忘掉之前的一份订单。真正的本事不是记住一切，而是决定*什么留在脑子里*、*什么卸载出去*。

每个 LLM 都运行在有限的上下文窗口内，也就是模型一次能「看到」的文本。无论窗口是 4000 还是 100 万 token，边界总在那里。边界内装着什么，决定了推理质量。

朴素做法把上下文窗口当成队列：最旧的消息从队尾掉出去。这是个糟糕的策略。对话开头的关键指令可能消失，无关的闲聊却还留着。这就是「lost in the middle」问题（Liu et al., 2023）：模型很难注意到埋在长上下文中部的信息。

工作记忆与上下文窗口管理借鉴认知心理学。Baddeley 的人类工作记忆模型描述了一个容量有限、带注意控制器的系统。这个控制器决定什么留在主动意识里，什么淡出。智能体需要同样的东西：一个管理层，负责对上下文元素打分、排定优先级、固定（保护）和淘汰（移除）。

**你将在这个 notebook 中构建：**
1. 一个 `ContextItem` 数据模型，带 token 计数和显著性得分。
2. 一个 `SalienceScorer`，按相关性、新近度和重要性给条目排序。
3. 一个 `EvictionEngine`，带两种策略：LRU 和重要性加权 LRU。
4. 一个 `ContextWindowManager`，统筹完整的生命周期。
5. 一个端到端示例，展示长对话中的上下文管理。

## 核心概念

- **上下文窗口（context window）**：LLM 一次能「看到」的文本。每个模型都有最大 token 上限。模型读取和推理的一切内容都必须装进这个窗口。
- **token**：模型处理的最小文本片段。"Chatbot" 可能是一个 token；"unbelievable" 可能是三个。token 数量决定成本和容量。
- **注意力管理（attention management）**：控制任意时刻由哪些信息占据上下文窗口。用有意的筛选取代被动累积。
- **上下文优先级排序（context prioritization）**：按与当前任务的相关性、使用的新近度和固有重要性，给每个上下文元素打分。
- **固定上下文（pinned context）**：无论如何都受保护、不被淘汰的条目。系统提示词、核心指令和用户偏好放在这里。把它们想成你桌上永久的便利贴。
- **动态淘汰策略（dynamic eviction policy）**：容量用尽后移除低价值上下文的策略。LRU（最近最少使用）移除最久未用的条目；按重要性淘汰则移除得分最低的条目。
- **显著性评分（salience scoring）**：算出一个数字，反映某个上下文条目对智能体*当前*推理步骤有多重要。我们会结合新近度衰减和重要性权重。
- **嵌入（embedding）**：一个数字列表（向量），捕捉文本的含义。含义相近的文本嵌入也相近。我们用嵌入度量相关性。
- **余弦相似度（cosine similarity）**：度量两个嵌入有多接近。取值从 -1（相反）到 1（含义相同）。
- **外部记忆库（external memory store）**：归档被淘汰条目的地方，之后仍可检索。条目离开活动上下文，但不会永远消失。

## 架构

<p align="center">
  <img src="../../images/diagrams/12_working_memory_context_window.svg" alt="工作记忆与上下文窗口架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    IS["Info Sources\n(User, Tools, Memory)"] --> SS["Salience Scorer"]
    SS --> PQ["Priority Queue"]
    PQ --> CW["Context Window"]

    subgraph CW["Context Window"]
        PZ["Pinned Zone\n(System prompt,\ncore instructions)"]
        DZ["Dynamic Zone\n(Scored & ranked\ncontext items)"]
    end

    CW --> LLM["LLM"]
    DZ --> EE["Eviction Engine"]
    EE --> EMS["External Memory Store"]
    EMS -->|"Retrieval on demand"| SS

    style PZ fill:#2d5a2d,stroke:#4a9,color:#fff
    style DZ fill:#2d4a7a,stroke:#49a,color:#fff
    style EE fill:#7a2d2d,stroke:#a44,color:#fff
```

</details>

**数据流**：来自不同来源的信息（用户消息、工具输出、检索到的记忆）进入**显著性评分器（Salience Scorer）**。评分器评估相关性和重要性，打过分的条目进入**优先级队列（Priority Queue）**。

**上下文窗口**分为两个区域。**固定区（Pinned Zone）**存放受保护的、始终在场的条目，比如系统提示词。**动态区（Dynamic Zone）**存放按显著性得分排序的条目。

当窗口接近容量上限时，**淘汰引擎（Eviction Engine）**从动态区移除得分最低的条目，并把它们归档到**外部记忆库（External Memory Store）**。当之前被淘汰的信息重新变得相关时，检索回路会把它送回显著性评分器。

## 环境准备

安装依赖。我们用到：
- `openai`：聊天补全和嵌入
- `tiktoken`：精确的 token 计数（这个库按模型实际使用的 token 切分文本）
- `numpy`：向量运算
- `python-dotenv`：从 `.env` 文件加载 API key

In [ ]:
%pip install -q tiktoken numpy python-dotenv deepagents langchain-openai

导入所有库并创建 API 客户端。你的 `.env` 文件里需要有 `LLM_API_KEY`。

In [ ]:
import os
import time
import json
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import tiktoken

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

### 第 1 步：`ContextItem` 数据模型

把 `ContextItem` 想成图书馆的卡片目录。每张卡片描述一本书，并带有元数据：书名、上次借出日期、被借过多少次。我们的 `ContextItem` 也是如此。它把一段文本和元数据包在一起，管理层靠这些元数据做淘汰决策。

每个条目记录自己的内容、角色、token 数、显著性得分、时间戳，以及是否被固定。

In [ ]:
def count_tokens(text: str) -> int:
    """用 tiktoken 计算 token 数。"""
    return len(ENCODER.encode(text))


@dataclass
class ContextItem:
    """上下文窗口里的单个条目，附带管理用元数据。"""
    content: str
    role: str  # 取值："system"、"user"、"assistant"、"tool"
    token_count: int = 0
    salience_score: float = 0.0
    created_at: float = 0.0
    last_accessed: float = 0.0
    access_count: int = 0
    pinned: bool = False
    source: str = "user"  # 取值：user、tool、memory、system
    item_id: str = ""
    embedding: Optional[list[float]] = field(default=None, repr=False)

    def __post_init__(self):
        if not self.token_count:
            self.token_count = count_tokens(self.content)
        if not self.created_at:
            self.created_at = time.time()
        if not self.last_accessed:
            self.last_accessed = self.created_at
        if not self.item_id:
            # 由内容 + 角色生成确定性 ID
            raw = f"{self.role}:{self.content}"
            self.item_id = hashlib.sha256(raw.encode()).hexdigest()[:12]

    def touch(self):
        """把该条目标记为最近访问。"""
        self.last_accessed = time.time()
        self.access_count += 1

    def to_message(self) -> dict:
        """转换成 OpenAI API 期望的字典格式。"""
        return {"role": self.role, "content": self.content}


# 快速测试
item = ContextItem(content="Hello, I'm Alice.", role="user", source="user")
print(f"Content: {item.content!r}")
print(f"Tokens:  {item.token_count}")
print(f"ID:      {item.item_id}")
print(f"Pinned:  {item.pinned}")

### 第 2 步：`SalienceScorer`

想象一位新闻编辑在决定哪些报道上头版。他们权衡三个因素：报道是否关于读者*今天*关心的事？（相关性）报道有多新鲜？（新近度）撇开时间不谈它有多重要？（重要性）我们的评分器用数字做同样的事。

显著性得分融合三个信号：
1. **嵌入相似度**：条目与当前查询有多贴近。我们计算两者嵌入之间的余弦相似度。
2. **新近度衰减**：最近被访问的条目得分更高。我们用指数衰减，让得分随时间快速下降。
3. **重要性权重**：给高优先级条目（比如工具结果或明确的用户目标）的静态加成。

公式：`score = w_rel * similarity + w_rec * recency + w_imp * importance`

In [ ]:
def get_embedding(text: str) -> list[float]:
    """从 OpenAI 获取一个嵌入向量。"""
    return embed_texts([text])[0]


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """计算两个向量的余弦相似度。"""
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(dot / norm)


# 按来源类型设定的重要性权重
SOURCE_IMPORTANCE = {
    "system": 1.0,
    "tool": 0.8,
    "memory": 0.6,
    "user": 0.5,
    "assistant": 0.4,
}

现在构建 `SalienceScorer` 类。它把三个信号合成一个数字：
相关性（与当前查询的嵌入相似度）、新近度（距上次访问的指数衰减）和重要性（按条目来源赋予的固定权重）。固定条目拿到无穷大的得分，永远不会被淘汰。

In [ ]:
class SalienceScorer:
    """按相关性、新近度和重要性给上下文条目打分。"""

    def __init__(
        self,
        weight_relevance: float = 0.5,
        weight_recency: float = 0.3,
        weight_importance: float = 0.2,
        recency_half_life: float = 300.0,  # 新近度减半所需的秒数
    ):
        self.w_rel = weight_relevance
        self.w_rec = weight_recency
        self.w_imp = weight_importance
        self.half_life = recency_half_life

    def _recency_score(self, item: ContextItem) -> float:
        """基于距上次访问时长的指数衰减。"""
        age = time.time() - item.last_accessed
        # decay = 0.5^(age / half_life)
        return float(np.exp(-0.693 * age / self.half_life))

    def _importance_score(self, item: ContextItem) -> float:
        """按条目来源给出的静态重要性。"""
        return SOURCE_IMPORTANCE.get(item.source, 0.3)

    def score(
        self,
        item: ContextItem,
        query_embedding: list[float],
    ) -> float:
        """计算单个条目的融合显著性得分。"""
        # 相关性：条目与当前查询的余弦相似度
        if item.embedding is None:
            item.embedding = get_embedding(item.content)
        relevance = cosine_similarity(item.embedding, query_embedding)
        # 截断到 [0, 1] 范围
        relevance = max(0.0, relevance)

        recency = self._recency_score(item)
        importance = self._importance_score(item)

        return (
            self.w_rel * relevance
            + self.w_rec * recency
            + self.w_imp * importance
        )

    def score_all(
        self,
        items: list[ContextItem],
        query_embedding: list[float],
    ) -> list[ContextItem]:
        """给所有条目打分并更新 salience_score 字段。"""
        for item in items:
            if not item.pinned:  # 固定条目保持最高得分
                item.salience_score = self.score(item, query_embedding)
            else:
                item.salience_score = float("inf")
        return items

我们用一个示例条目和查询来测试评分器。这个条目直接回答了查询，所以得分应该很高。

In [ ]:
# 快速测试
scorer = SalienceScorer()
test_item = ContextItem(
    content="The capital of France is Paris.",
    role="assistant",
    source="assistant",
)
query_emb = get_embedding("What is the capital of France?")
score = scorer.score(test_item, query_emb)
print(f"Salience score: {score:.4f}")

### 第 3 步：`EvictionEngine`

想象一辆拥挤的公交车。新乘客上车而座位已满时，得有人站着，或者有人下车。一个公平的规则是：坐得最久的人让座（LRU）。更聪明的规则是：目的地最近的人先下车（按重要性加权）。我们的淘汰引擎两种策略都实现了。

引擎从未固定的条目中挑选移除对象，直到为新内容腾出足够 token。它支持两种策略：
- **LRU**：优先移除 `last_accessed` 时间戳最旧的条目。
- **重要性加权 LRU**：混合新近度和显著性得分。又旧*又*低分的条目先走。

In [ ]:
class EvictionEngine:
    """上下文窗口已满时挑选要淘汰的条目。"""

    def __init__(self, policy: str = "importance_weighted_lru"):
        self.policy = policy

    def select_victims(
        self,
        items: list[ContextItem],
        tokens_to_free: int,
    ) -> list[ContextItem]:
        """挑选未固定的条目进行淘汰，至少腾出 tokens_to_free 个 token。"""
        candidates = [item for item in items if not item.pinned]

        if self.policy == "lru":
            # 按 last_accessed 升序排序（最旧的在前）
            candidates.sort(key=lambda x: x.last_accessed)
        elif self.policy == "importance_weighted_lru":
            # 按 salience_score 升序排序（得分最低的先淘汰）
            candidates.sort(key=lambda x: x.salience_score)
        else:
            raise ValueError(f"Unknown eviction policy: {self.policy}")

        victims = []
        freed = 0
        for candidate in candidates:
            if freed >= tokens_to_free:
                break
            victims.append(candidate)
            freed += candidate.token_count

        return victims


# 快速测试
evictor = EvictionEngine(policy="importance_weighted_lru")
test_items = [
    ContextItem(content="Old low-score item", role="assistant", salience_score=0.1),
    ContextItem(content="Recent high-score item", role="user", salience_score=0.9),
    ContextItem(content="Pinned system prompt", role="system", pinned=True),
]
victims = evictor.select_victims(test_items, tokens_to_free=10)
print(f"Evicting {len(victims)} item(s): {[v.content for v in victims]}")

### 第 4 步：外部记忆库

把外部记忆库想成书桌旁的文件柜。桌子（上下文窗口）太乱时，你把文件移进柜子。它们没有消失，需要时随时取回。

我们的外部记忆库连同嵌入一起归档被淘汰的条目。当评分器发现某个旧条目重新变得相关，就由记忆库把它取回。

In [ ]:
class ExternalMemoryStore:
    """归档被淘汰的上下文条目，供日后检索。"""

    def __init__(self):
        self.archive: list[ContextItem] = []

    def store(self, item: ContextItem) -> None:
        """归档一个被淘汰的条目。"""
        # 确保条目带有嵌入，便于日后检索
        if item.embedding is None:
            item.embedding = get_embedding(item.content)
        self.archive.append(item)

    def retrieve(
        self,
        query_embedding: list[float],
        top_k: int = 3,
        min_similarity: float = 0.3,
    ) -> list[ContextItem]:
        """找出与当前查询最相关的归档条目。"""
        if not self.archive:
            return []

        scored = []
        for item in self.archive:
            sim = cosine_similarity(item.embedding, query_embedding)
            if sim >= min_similarity:
                scored.append((sim, item))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in scored[:top_k]]

    def remove(self, item_id: str) -> None:
        """从归档中移除条目（比如重新插入之后）。"""
        self.archive = [i for i in self.archive if i.item_id != item_id]

    def __len__(self) -> int:
        return len(self.archive)


print("ExternalMemoryStore ready.")

### 第 5 步：`ContextWindowManager`

这是整个系统的指挥。它协调评分器、淘汰器和记忆库。每一轮它都要：
1. 用当前查询给所有动态条目打分。
2. 检查新条目放不放得下。放不下就淘汰得分最低的条目。
3. 当归档条目匹配当前查询时，检索它们。
4. 组装最终提示词：固定条目在前，然后是按得分排序的动态条目。

In [ ]:
class ContextWindowManager:
    """统筹打分、插入、淘汰和提示词组装。"""

    def __init__(
        self,
        max_tokens: int = 4000,
        system_prompt: str = "You are a helpful assistant.",
        eviction_policy: str = "importance_weighted_lru",
        retrieval_top_k: int = 2,
    ):
        self.max_tokens = max_tokens
        self.scorer = SalienceScorer()
        self.evictor = EvictionEngine(policy=eviction_policy)
        self.external_store = ExternalMemoryStore()
        self.retrieval_top_k = retrieval_top_k

        # 固定条目永远留在窗口里
        self.pinned_items: list[ContextItem] = []
        # 动态条目竞争空间
        self.dynamic_items: list[ContextItem] = []

        # 记录淘汰历史，供示例展示
        self.eviction_log: list[dict] = []

        # 把系统提示词设为固定条目
        sys_item = ContextItem(
            content=system_prompt,
            role="system",
            pinned=True,
            source="system",
        )
        self.pinned_items.append(sys_item)

    def _current_token_usage(self) -> int:
        """窗口里当前的 token 总数。"""
        pinned = sum(item.token_count for item in self.pinned_items)
        dynamic = sum(item.token_count for item in self.dynamic_items)
        return pinned + dynamic

    def _available_tokens(self) -> int:
        """可用于新内容的 token 数。"""
        return self.max_tokens - self._current_token_usage()

    def _try_retrieve_from_archive(
        self, query_embedding: list[float]
    ) -> list[ContextItem]:
        """检查外部存储里相关的被淘汰条目。"""
        retrieved = self.external_store.retrieve(
            query_embedding, top_k=self.retrieval_top_k
        )
        return retrieved

`add_item` 方法是真正干活的地方。它给新条目打分，用新查询给现有条目重新打分，空间不够时淘汰低分条目，并检查归档里有没有相关的被淘汰内容可以取回。

In [ ]:
def _add_item(
    self,
    content: str,
    role: str,
    source: str = "user",
    query_text: str | None = None,
) -> None:
    """向上下文窗口添加新条目，必要时执行淘汰。"""
    item = ContextItem(content=content, role=role, source=source)
    query = query_text or content
    query_embedding = get_embedding(query)

    # 给新条目打分
    item.salience_score = self.scorer.score(item, query_embedding)
    item.embedding = get_embedding(item.content)

    # 用新查询给所有现有动态条目重新打分
    self.scorer.score_all(self.dynamic_items, query_embedding)

    # 检查是否需要淘汰
    tokens_needed = item.token_count
    available = self._available_tokens()

    if tokens_needed > available:
        tokens_to_free = tokens_needed - available
        victims = self.evictor.select_victims(
            self.dynamic_items, tokens_to_free
        )
        for victim in victims:
            self.external_store.store(victim)
            self.dynamic_items.remove(victim)
            self.eviction_log.append({
                "evicted": victim.content[:60],
                "score": round(victim.salience_score, 4),
                "tokens_freed": victim.token_count,
            })

    # Try to retrieve relevant archived items
    retrieved = self._try_retrieve_from_archive(query_embedding)
    for ret_item in retrieved:
        # 有空间才重新插入
        if ret_item.token_count <= self._available_tokens() - tokens_needed:
            ret_item.touch()
            self.dynamic_items.append(ret_item)
            self.external_store.remove(ret_item.item_id)

    # 插入新条目
    self.dynamic_items.append(item)

ContextWindowManager.add_item = _add_item


最后，`build_messages` 为 LLM 组装提示词。固定条目在前，动态条目按时间顺序排列。`status` 方法给出 token 用量和条目数量的快照。

In [ ]:
def _build_messages(self) -> list[dict]:
    """为 LLM 组装最终的消息列表。"""
    messages = []

    # 固定条目在前（系统提示词等）
    for item in self.pinned_items:
        item.touch()
        messages.append(item.to_message())

    # 动态条目按创建时间排序，保持对话顺序
    ordered = sorted(self.dynamic_items, key=lambda x: x.created_at)
    for item in ordered:
        item.touch()
        messages.append(item.to_message())

    return messages

def _status(self) -> dict:
    """返回管理器状态的快照。"""
    return {
        "max_tokens": self.max_tokens,
        "used_tokens": self._current_token_usage(),
        "available_tokens": self._available_tokens(),
        "pinned_items": len(self.pinned_items),
        "dynamic_items": len(self.dynamic_items),
        "archived_items": len(self.external_store),
        "total_evictions": len(self.eviction_log),
    }


print("ContextWindowManager ready.")

ContextWindowManager.build_messages = _build_messages

ContextWindowManager.status = _status


### 第 6 步：聊天函数

现在把管理器接到 LLM 上。`chat` 函数添加用户消息、构建托管提示词、调用 API，然后存储回复。

In [ ]:
def chat_with_managed_context(
    manager: ContextWindowManager,
    user_input: str,
    verbose: bool = False,
) -> str:
    """通过托管的上下文窗口发送一条消息。"""
    # 把用户消息加入托管上下文
    manager.add_item(
        content=user_input,
        role="user",
        source="user",
    )

    # 从托管上下文构建提示词
    messages = manager.build_messages()

    if verbose:
        status = manager.status()
        print(f"  [Context: {status['used_tokens']}/{status['max_tokens']} tokens, "
              f"{status['dynamic_items']} dynamic, "
              f"{status['archived_items']} archived]")

    # 调用 LLM
    response = llm.invoke(messages)

    assistant_text = to_text(response.content)

    # 把助手回复加入托管上下文
    manager.add_item(
        content=assistant_text,
        role="assistant",
        source="assistant",
        query_text=user_input,  # 用用户的查询来打分
    )

    return assistant_text


print("Chat function ready.")

## 运行示例

我们来看管理器的实际表现。我们用一个很小的 token 预算（800 token）来快速触发淘汰。真实应用里，你应该把它设成模型上下文上限减去为回复预留的部分。

我们将进行一段规划旅行的对话。早期消息会随着新消息到来被淘汰。然后我们会问起某条已被淘汰消息里的内容，看看检索回路如何工作。

In [ ]:
# 用很紧的 token 预算创建管理器，观察淘汰的实际效果
manager = ContextWindowManager(
    max_tokens=800,
    system_prompt="You are a travel planning assistant. Keep replies under 3 sentences.",
    eviction_policy="importance_weighted_lru",
)

print(f"Starting state: {manager.status()}")
print("=" * 60)

conversation = [
    "Hi! I'm planning a trip to Japan in October.",
    "My budget is about $3,000 for two weeks.",
    "I love hiking and traditional temples.",
    "I'm also interested in the food scene in Osaka.",
    "What's the weather like in Kyoto in October?",
    "Can you recommend a good ryokan near Kyoto?",
    "I'm vegetarian, will that be a problem?",
    "What was my budget again?",  # 测试对被淘汰信息的检索
]

for msg in conversation:
    print(f"\nUser: {msg}")
    reply = chat_with_managed_context(manager, msg, verbose=True)
    print(f"Agent: {reply}")

我们来检查最终状态：哪些条目还在活动窗口里，哪些被淘汰了，以及淘汰日志。

In [ ]:
print("Final status:")
for key, value in manager.status().items():
    print(f"  {key}: {value}")

print("\n--- Active context (dynamic items) ---")
for item in sorted(manager.dynamic_items, key=lambda x: x.created_at):
    label = item.role.upper()
    preview = item.content[:70] + ("..." if len(item.content) > 70 else "")
    print(f"  [{label}] score={item.salience_score:.3f} tokens={item.token_count}: {preview}")

print(f"\n--- Eviction log ({len(manager.eviction_log)} events) ---")
for entry in manager.eviction_log:
    print(f"  Evicted: {entry['evicted']!r} (score={entry['score']}, freed={entry['tokens_freed']} tokens)")

print(f"\n--- External store ({len(manager.external_store)} items) ---")
for item in manager.external_store.archive:
    preview = item.content[:70] + ("..." if len(item.content) > 70 else "")
    print(f"  [{item.role.upper()}] {preview}")

### 比较淘汰策略

我们把同一段对话分别用纯 LRU 和重要性加权 LRU 跑一遍，看看策略的选择如何影响窗口里留下什么。

In [ ]:
def run_conversation_with_policy(policy: str) -> dict:
    """跑同一段对话并返回最终状态。"""
    mgr = ContextWindowManager(
        max_tokens=800,
        system_prompt="You are a travel planning assistant. Keep replies under 3 sentences.",
        eviction_policy=policy,
    )

    test_messages = [
        "Hi! I'm planning a trip to Japan in October.",
        "My budget is about $3,000 for two weeks.",
        "I love hiking and traditional temples.",
        "I'm also interested in the food scene in Osaka.",
        "What's the weather like in Kyoto in October?",
        "Can you recommend a good ryokan near Kyoto?",
    ]

    for msg in test_messages:
        chat_with_managed_context(mgr, msg)

    return {
        "policy": policy,
        "status": mgr.status(),
        "eviction_count": len(mgr.eviction_log),
        "active_items": [
            item.content[:50] for item in mgr.dynamic_items
        ],
        "archived_count": len(mgr.external_store),
    }


for policy in ["lru", "importance_weighted_lru"]:
    result = run_conversation_with_policy(policy)
    print(f"\nPolicy: {result['policy']}")
    print(f"  Evictions: {result['eviction_count']}")
    print(f"  Active items: {len(result['active_items'])}")
    print(f"  Archived: {result['archived_count']}")
    print(f"  Active content:")
    for content in result["active_items"]:
        print(f"    - {content}...")

## 权衡

### 这项技术的优势

- **长会话**：数百轮的对话保持连贯。模型始终看到最相关的上下文，而不是最新的。
- **可预测的 token 预算**：你精确控制每次调用送给 LLM 多少 token。成本和延迟都不会有意外。
- **关键指令保留**：固定条目保证系统提示词和关键目标不会被聊天内容挤出去。
- **优雅降级**：容量耗尽时，系统移除价值最低的条目，而不是从对话中间截断。

### 什么时候会失效

- **评分开销**：每条新消息都触发嵌入调用和重新打分。快节奏对话下这会增加延迟。在我们的实现里，每轮需要 1-3 次嵌入 API 调用。
- **显著性估计不完美**：嵌入度量语义相似度，不是逻辑重要性。一条消息可能语义上离当前查询很远，逻辑上却至关重要（比如一边讨论晚饭计划，一边说「我的航班早上 6 点起飞」）。
- **对话流丢失**：被淘汰的消息在对话历史里留下空隙。模型看不到完整脉络，回复可能显得前后脱节。
- **复杂度成本**：朴素缓冲记忆只要 20 行代码。这个方案还要加上评分、淘汰、归档和检索，出 bug 的地方更多。

## 延伸阅读

- Liu et al., ["Lost in the Middle: How Language Models Use Long Contexts,"](https://arxiv.org/abs/2307.03172) 2023. LLM 难以处理放在长上下文中部的信息。这项研究是显式上下文管理的动机来源。
- Baddeley, ["The Episodic Buffer: A New Component of Working Memory?"](https://doi.org/10.1016/S1364-6613%2800%2901538-2) *Trends in Cognitive Sciences*, 2000. 奠基性的认知模型：带注意控制的有限容量工作记忆。
- Mu et al., ["Learning to Compress Prompts with Gist Tokens,"](https://arxiv.org/abs/2304.08467) 2023. 教模型把冗长的提示词压缩成紧凑的 gist token，让同一个窗口装下更多内容。
- Modarressi et al., ["RET-LLM: Towards a General Read-Write Memory for LLMs,"](https://arxiv.org/abs/2305.14322) 2023. 为 LLM 提出显式的读写记忆层，与上下文窗口管理互补。
- [OpenAI Embeddings 指南](https://platform.openai.com/docs/guides/embeddings)：本 notebook 使用的嵌入模型的细节。
- [GitHub 上的 tiktoken](https://github.com/openai/tiktoken)：做 token 计数的库。

---

*\u2190 上一篇：[11 - 程序性记忆](../11_procedural_memory/) \u00b7 下一篇：[13 - 分层记忆](../13_hierarchical_memory_layers/) \u2192*

## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：用户固定条目
给 `ContextItem` 加一个 `pinned` 布尔字段。固定条目永远不能被 `EvictionEngine` 淘汰。修改 `ContextWindowManager`，支持 `pin(item_id)` 方法。测试方法：固定一个关键事实，跑一段通常会把这条事实淘汰的长对话，看它是否留得住。

### 挑战 2：淘汰策略比较
用 LRU 淘汰和重要性加权淘汰各跑一次 50 轮对话。每次跑完后问 10 个回忆型问题并给答案打分。比较哪种策略保留了更多有用信息，报告两者的查全率差异。

### 挑战 3：带向量检索的外部记忆
把基础的 `ExternalMemoryStore` 换成基于 ChromaDB 的存储，支持基于嵌入的检索。查询到来时，检查外部存储里有没有相关的被淘汰条目，把它们提升回上下文窗口。这为技术 13 分层记忆的分层架构搭好了桥。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--12-working-memory-context-window--working-memory-context-window)
